# Global Health Progress, 2000–2019: A Data Story from the World Bank's Health, Nutrition and Population Statistics

**3520872** | CSCU9M3 — Scripting for Data Science | Autumn 2026

In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Colab always has a module called google.colab loaded automatically, so I
# can use that to check if the notebook is running in Colab or not, and
# point at the right folder for the data either way.
running_in_colab = "google.colab" in sys.modules

if running_in_colab:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/Untitled folder/data/hnp_stats"
else:
    DATA_DIR = "data/hnp_stats"

## Executive Summary

This report uses the World Bank's Health, Nutrition and Population Statistics data to ask whether global health improved between 2000 and 2019, and whether that improvement was shared equally. After removing regional and income-group aggregates and restricting to five indicators with good coverage, 217 countries remain. Checking the data first shows health expenditure is missing more often than the other four, regions vary hugely in size, from 3 countries in North America to 56 in Europe & Central Asia, and life expectancy and fertility rate are skewed in opposite directions, so the median is more representative for both. The three questions that follow find under-5 mortality fell by a similar percentage across all four income groups, between 40% and 53%, so poorer countries have not lagged. Health expenditure and life expectancy correlate only moderately at 0.44, spending explains part of the gap but not most of it. Measles immunisation converged across regions, the two already near full coverage, North America and Latin America & Caribbean, both lost a little ground, while South Asia and Sub-Saharan Africa caught up the most.

## Introduction

This report uses the World Bank's Health, Nutrition and Population Statistics dataset, a country-by-year record of health, nutrition and population indicators going back to 1960, covering over 200 countries. It was chosen for its size and structure, a single file holding decades of comparable measurements across almost every country, enough to support a real data story. The question behind this report, has global health improved since 2000 and has that improvement reached richer and poorer countries equally, is one most people have an intuition about before looking at any numbers. Before analysing the data I expected health outcomes to have improved nearly everywhere, given the well known global gains in child mortality and vaccination coverage, but I also expected poorer countries to be catching up more slowly, since they have fewer resources to spend on health, and I expected health spending to be closely tied to life expectancy. The findings below show a more mixed picture on both counts, which is exactly what descriptive analytics is for, testing an intuition against the actual numbers.

## Data Cleaning

The World Bank's Health, Nutrition and Population Statistics (HNP) data comes as linked CSV files, a main data file (one row per country/indicator, one column per year), a country metadata file, and an indicator metadata file. The main file mixes real countries with regional and income-group aggregates, holds hundreds of indicators and decades of unused years, and has many combinations missing data for the years that matter.

In [ ]:
data = pd.read_csv(f"{DATA_DIR}/HNP_StatsData.csv")
country_info = pd.read_csv(f"{DATA_DIR}/HNP_StatsCountry.csv")

# Both files end every line with a trailing comma, so pandas reads an extra
# empty column called "Unnamed: xx" at the end. I get rid of those columns
# before doing anything else with the data.
extra_columns = [col for col in data.columns if col.startswith("Unnamed")]
data = data.drop(columns=extra_columns)

extra_columns = [col for col in country_info.columns if col.startswith("Unnamed")]
country_info = country_info.drop(columns=extra_columns)

print("Main data file:", data.shape)
print("Country info file:", country_info.shape)

Removing non-country aggregates. Rows such as "Africa Eastern and Southern" are regional/income-group aggregates, not countries, and would distort any country comparison. Hand-picking aggregate names to drop would be brittle, missing any mis-spelled or renamed. The country metadata file already distinguishes real countries, every real country has a Region value, every aggregate a blank one, so filtering on that column is a rule based on the data rather than a manual list.

In [ ]:
# Rows with no Region are not real countries, they are totals for a whole
# region or income group (e.g. "Africa Eastern and Southern"). I build a
# dictionary from Country Code to Region, and another to Income Group,
# using the real countries only, then look up each row's code in them.
real_countries = country_info[country_info["Region"].notna()]
country_codes = list(real_countries["Country Code"])
regions = list(real_countries["Region"])
income_groups = list(real_countries["Income Group"])

code_to_region = {}
code_to_income = {}
for i in range(len(country_codes)):
    code_to_region[country_codes[i]] = regions[i]
    code_to_income[country_codes[i]] = income_groups[i]

data["Region"] = [code_to_region.get(code) for code in data["Country Code"]]
data["Income Group"] = [code_to_income.get(code) for code in data["Country Code"]]

# A code that was not in the dictionary (a region/income total, not a
# real country) ends up with a Region of None here, so this drops them.
data = data[data["Region"].notna()]

print("Rows after removing region/income totals:", len(data))
print("Number of countries left:", data["Country Code"].nunique())

Choosing indicators. The full dataset covers hundreds of indicators, far more than this report can cover, so five were chosen to address one question, has global health improved since 2000, and is it shared equally. The five, life expectancy, under-5 mortality, measles immunisation, fertility rate, and health expenditure, were chosen for good coverage and because together they cover outcomes, a driver, and context.

In [ ]:
indicators_wanted = {
    "SP.DYN.LE00.IN": "Life expectancy at birth (years)",
    "SH.DYN.MORT": "Under-5 mortality (per 1,000 live births)",
    "SH.IMM.MEAS": "Measles immunisation (% of children 12-23 months)",
    "SP.DYN.TFRT.IN": "Fertility rate (births per woman)",
    "SH.XPD.CHEX.GD.ZS": "Current health expenditure (% of GDP)",
}

data = data[data["Indicator Code"].isin(indicators_wanted)]
data["Indicator Name"] = data["Indicator Code"].map(indicators_wanted)
print("Rows after keeping only the 5 indicators I want:", len(data))

Choosing the year range. Health expenditure is only reported from 2000, so an earlier start would mean comparing on four indicators for most of the window and five only at the end, hence 2000. 2019 is the end point rather than 2020-2021, since COVID-19 disrupted both health outcomes and data collection, making this partly about the pandemic rather than the longer-run trend this report is about.

In [ ]:
YEARS = [str(year) for year in range(2000, 2020)]
columns_to_keep = ["Country Name", "Country Code", "Region", "Income Group",
                    "Indicator Name", "Indicator Code"] + YEARS
data = data[columns_to_keep]
print("Columns kept:", list(data.columns))

Handling missing values. A row is only usable for the before/after comparisons later if it has a value in both 2000 and 2019. Rows missing either are dropped rather than interpolated, since interpolating would invent a trend for a country that never reported reliably, and that gap is a real feature of the data, not something to hide.

In [ ]:
missing_2000 = data["2000"].isna().sum()
missing_2019 = data["2019"].isna().sum()
print("Rows missing a 2000 value:", missing_2000)
print("Rows missing a 2019 value:", missing_2019)

clean_data = data.dropna(subset=["2000", "2019"])
clean_data = clean_data.reset_index(drop=True)

percent_kept = round(len(clean_data) / len(data) * 100)
print("Rows kept:", len(clean_data), "out of", len(data), f"({percent_kept}%)")

In [ ]:
clean_data[["Country Name", "Region", "Income Group", "Indicator Name", "2000", "2019"]].head()

clean_data is the dataset used for the rest of this report, real countries only, five indicators, years 2000–2019, with both endpoint years present. Intermediate years may still have gaps, which is fine since only the EDA section uses them, as year-by-year averages rather than individual values.

## Exploratory Data Analysis

This section checks what clean_data contains, how much of each indicator is missing, how the 217 countries split across regions, and what a typical indicator's spread looks like.

In [ ]:
# I check missingness on data (the 1085 rows before I dropped the ones
# missing an endpoint year), grouped by indicator, so I can see if the
# missing values are spread evenly or concentrated in one indicator.
rows_missing_an_endpoint = data[data["2000"].isna() | data["2019"].isna()]
missing_by_indicator = rows_missing_an_endpoint.groupby("Indicator Name").size()
total_by_indicator = data.groupby("Indicator Name").size()

missing_percent = (missing_by_indicator / total_by_indicator * 100).round(1)
print(missing_percent)

plt.bar(missing_percent.index, missing_percent.values)
plt.ylabel("Percent of countries missing a 2000 or 2019 value")
plt.xticks(rotation=45, ha="right")
plt.show()

Checking missing data by indicator. Current health expenditure is missing for 37 of 217 countries, almost double the rate of life expectancy and fertility rate, each missing only 18, since it is newer with less historical reporting. Any result using it rests on a smaller set of countries than the other four.

In [ ]:
countries_per_region = clean_data.groupby("Region")["Country Code"].nunique()
countries_per_region = countries_per_region.sort_values()
print(countries_per_region)

plt.bar(countries_per_region.index, countries_per_region.values)
plt.ylabel("Number of countries")
plt.xticks(rotation=45, ha="right")
plt.show()

Checking country counts by region. The regions are far from equal, North America has 3 countries and South Asia 8, while Europe & Central Asia has 56. A difference between two small regions could come from one or two countries moving, so regional comparisons later lean more on the larger regions.

In [ ]:
life_expectancy_2019 = clean_data[clean_data["Indicator Name"] == "Life expectancy at birth (years)"]["2019"]
fertility_2019 = clean_data[clean_data["Indicator Name"] == "Fertility rate (births per woman)"]["2019"]

print("Life expectancy, mean:", round(life_expectancy_2019.mean(), 1), "median:", round(life_expectancy_2019.median(), 1))
print("Fertility rate, mean:", round(fertility_2019.mean(), 2), "median:", round(fertility_2019.median(), 2))

plt.hist(life_expectancy_2019, bins=15)
plt.xlabel("Life expectancy at birth in 2019 (years)")
plt.ylabel("Number of countries")
plt.show()

plt.hist(fertility_2019, bins=15)
plt.xlabel("Fertility rate in 2019 (births per woman)")
plt.ylabel("Number of countries")
plt.show()

Checking the shape of two indicators. Life expectancy has a mean of 73.0 against a median of 74.2, a smaller group of countries with much lower life expectancy pulls the mean down. Fertility rate is the opposite, a mean of 2.62 against a median of 2.19, pulled up by a smaller group of high fertility countries. The median is therefore more representative for a typical country on both, and descriptive analytics uses that where it matters.

## Descriptive Analytics

This section answers three questions using clean_data. Two compare a 2000 value against a 2019 value within a group, Income Group and Region, so one function, average_by_group, is used twice rather than repeating the same groupby code.

In [ ]:
def average_by_group(indicator_name, group_column):
    # Takes the rows for one indicator, groups them by whichever column I
    # pass in (Income Group or Region), and works out the average 2000
    # value, the average 2019 value, and how much it changed.
    subset = clean_data[clean_data["Indicator Name"] == indicator_name]
    grouped = subset.groupby(group_column)[["2000", "2019"]].mean()
    grouped["change"] = grouped["2019"] - grouped["2000"]
    return grouped.round(1)

### Has under-5 mortality fallen equally across income groups?

Child mortality under 5 is one of the clearest signs a health system is working, so whether poorer countries are catching up or falling further behind says something real about where progress is happening.

In [ ]:
mortality_by_income = average_by_group("Under-5 mortality (per 1,000 live births)", "Income Group")
mortality_by_income = mortality_by_income.sort_values("2000", ascending=False)
print(mortality_by_income)

# The change column on its own makes low income countries look like they
# did most of the improving, but that is partly just because they started
# so much higher. Dividing by the 2000 value puts all four groups on the
# same scale, as a percent change instead of a raw one.
percent_change = (mortality_by_income["change"] / mortality_by_income["2000"] * 100).round(0)
print("\npercent change:")
print(percent_change)

plt.bar(percent_change.index, percent_change.values)
plt.ylabel("Percent change in under-5 mortality, 2000 to 2019")
plt.xticks(rotation=45, ha="right")
plt.show()

Low income countries cut under-5 mortality from 149.0 to 69.8, a 53% drop, the largest despite starting highest. High income countries cut theirs from 10.3 to 6.1, a 40% drop, the smallest. All four land between 40% and 53%, so the poorest have not improved more slowly, if anything slightly faster.

### Is health spending related to how long people live?

This matters for where health budgets go, since a weak link would mean raising spending alone is unlikely to move life expectancy much.

In [ ]:
life_expectancy = clean_data[clean_data["Indicator Name"] == "Life expectancy at birth (years)"]
life_expectancy = life_expectancy[["Country Code", "2019"]].rename(columns={"2019": "life_expectancy"})

health_spending = clean_data[clean_data["Indicator Name"] == "Current health expenditure (% of GDP)"]
health_spending = health_spending[["Country Code", "2019"]].rename(columns={"2019": "health_spending"})

# I only use 2019 here instead of the whole 2000 to 2019 window, because
# health expenditure only starts in 2000. This keeps the comparison to
# countries with a real value in the most recent year, rather than mixing
# different years together.
both = life_expectancy.merge(health_spending, on="Country Code", how="inner")
print("Countries with both values:", len(both))

correlation = both["life_expectancy"].corr(both["health_spending"])
print("Correlation:", round(correlation, 2))

plt.scatter(both["health_spending"], both["life_expectancy"])
plt.xlabel("Health expenditure in 2019 (% of GDP)")
plt.ylabel("Life expectancy in 2019 (years)")
plt.show()

Across the 171 countries with both figures, health expenditure and life expectancy correlate at 0.44, a moderate relationship. Higher spending goes with longer life expectancy on average, but explains only part of the gap, income and the kind of care that money buys likely do the rest.

### Which regions improved measles immunisation the most, and which fell behind?

Measles immunisation is a basic marker of how far routine childhood healthcare reaches, so a region moving backwards is a different problem to one simply slower to catch up.

In [ ]:
immunisation_by_region = average_by_group("Measles immunisation (% of children 12-23 months)", "Region")
immunisation_by_region = immunisation_by_region.sort_values("change")
print(immunisation_by_region)

plt.bar(immunisation_by_region.index, immunisation_by_region["change"])
plt.ylabel("Change in measles immunisation, 2000 to 2019 (percentage points)")
plt.xticks(rotation=45, ha="right")
plt.show()

South Asia gained the most, up 20.4 points from 70.1% to 90.5%, and Sub-Saharan Africa gained 16.0. North America is the only region that fell, and Latin America & Caribbean slipped slightly too. The regions closest to full coverage lost ground, while the regions furthest behind caught up the most, since a region already above 90% has little room to grow, so a small dip there is coverage slipping rather than room running out.

## Suggestions

This section suggests ways to extend this report. One is already possible with the files that came with the dataset and is shown below, the other three need more data or modelling than this report covers, so they are explained rather than built.

In [ ]:
series_info = pd.read_csv(f"{DATA_DIR}/HNP_StatsSeries.csv")
extra_columns = [col for col in series_info.columns if col.startswith("Unnamed")]
series_info = series_info.drop(columns=extra_columns)
series_info = series_info.rename(columns={"Series Code": "Indicator Code"})

# clean_data never uses the Source column, even though it came with the
# dataset in HNP_StatsSeries.csv, so I link it in here to see where each
# of the five indicators actually comes from.
indicator_sources = clean_data[["Indicator Code", "Indicator Name"]].drop_duplicates()
indicator_sources = indicator_sources.merge(series_info[["Indicator Code", "Source"]], on="Indicator Code")

for _, row in indicator_sources.iterrows():
    print(row["Indicator Name"])
    print("  source:", row["Source"][:90])

The five indicators come from three organisations, the UN Inter-agency Group for Child Mortality Estimation, WHO and UNICEF, the WHO Global Health Expenditure database, and the UN Population Division. This is already in HNP_StatsSeries.csv, the metadata file that came with the dataset, but nothing earlier in this report uses it. This matters because indicators from different organisations can use different survey methods and update schedules, so comparing across indicators is partly comparing across data collection systems too.

Further suggestions, not built here. Merging in the World Bank's Gender Statistics data, sharing the same Country Code and year columns as HNP, would let gender-equality indicators be checked against the outcomes above, for example whether higher female school enrolment goes with faster drops in under-5 mortality. Once Seaborn and Plotly are covered later in the module, the bar charts above could become a single map, carrying more information in one view than seven bars. And a model taking life expectancy, fertility rate, and health spending together, rather than spending against life expectancy alone, could show whether spending still matters once fertility is accounted for.

## Bibliography

_[Phase 6 — to be completed last.]_